# EX1:
The n-dimensional tensor mastery challenge: Combine the `Head` and `MultiHeadAttention` into one class that processes all the heads in parallel, treating the heads as another batch dimension (answer is in nanoGPT).

In [4]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [5]:
batch_size = 4
block_size = 8
n_embd = 32
dropout = 0.2

In [ ]:

class Head(nn.Module):
    """ one head of self-attention"""

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)     # (B, T, head_size)
        # (num_heads, B, T, head_size)
        q = self.query(x)   # (B, T, head_size)
        # (num_heads, B, T, head_size)

        # compute attention scores - affinities
        wei = q @ k.transpose(-2, -1) * C**-0.5 # (B, T, head_size) @ (B, head_size, T) -> (B, T, T)
        # (num_heads, B, T, head_size) @ (num_heads, B, head_size, T) -> (num_heads, B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))    # (B, T, T)
        wei = F.softmax(wei, dim=-1)    # (B, T, T)
        wei = self.dropout(wei)
    
        # perform the weighted aggregation of the values
        v = self.value(x)   # (B, T, head_size)
        # (num_heads, B, T, head_size)
        out = wei @ v   # (B, T, T) @ (B, T, head_size) -> (B, T, head_size)
        # (B, T, T) @ (num_heads, B, T, head_size) -> (num_heads, B, T, head_size)
        # TODO: (num_heads, B, T, head_size) -> (B, T, num_heads * head_size)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel"""

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

In [ ]:
x = torch.randn((batch_size, block_size, n_embd)).float()
h = Head(n_embd // 4)
mh = MultiHeadAttention(4, n_embd // 4)
out = mh(x)
print('h(x).shape:', h(x).shape)
print('mh(x).shape:', out.shape)
# (4, 8, 8) + (4, 8, 8) + (4, 8, 8) + (4, 8, 8) = (4, 8, 32)

h(x).shape: torch.Size([4, 8, 8])
mh(x).shape: torch.Size([4, 8, 32])
